In [ ]:
import importlib
import logging
import multiprocessing
import os
import random
import time

import jax
import lightning_xpu
import lightning
import torch

In [ ]:
# "COMPOSITE": single device per GPU card.
# "FLAT" : single device per GPU stack; two devices per GPU card.
modes = ["FLAT", "COMPOSITE"]

In [ ]:
def run_in_subprocess(target=None, *args, **kwargs):
    subprocess = multiprocessing.Process(target=target, args=args, kwargs=kwargs)
    subprocess.start()
    subprocess.join()

In [ ]:
def pytorch_check_devices(mode="FLAT"):
    
    os.environ["ZE_FLAT_DEVICE_HIERARCHY"] = mode

    # Define device types to be considered.
    device_types = ["cpu", "cuda", "mps", "xpu", "fictional_device"]
    
    # Print information about available device types.
    print(f"{mode} mode - devices seen by torch:")
    for device_type in sorted(device_types):
        # Determine number of devices of each type.
        try:
            device_module = importlib.import_module(f"torch.{device_type}")
        except ModuleNotFoundError:
            device_module = None
        n_device = getattr(device_module, "device_count", lambda: 0)()
        devices = [f"{device_type}:{idx}" for idx in range(n_device)]
        print(f"    {device_type}: {devices}")
    print()

for mode in modes:
    run_in_subprocess(target=pytorch_check_devices, mode=mode)

In [ ]:
def lightning_check_devices(mode="FLAT"):
    from lightning.pytorch.accelerators import AcceleratorRegistry
    os.environ["ZE_FLAT_DEVICE_HIERARCHY"] = mode
    print(f"{mode} mode - devices seen by lightning:")
    for device_type in sorted(AcceleratorRegistry.available_accelerators()):
        try:
            device = AcceleratorRegistry.get(device_type)
        except ModuleNotFoundError:
            device = None
        devices = (device.get_parallel_devices(device.auto_device_count())
                   if getattr(device, "is_available", lambda: False)() else [])
        print(f"    {device_type}: {devices}")
    print()

for mode in modes:
    run_in_subprocess(target=lightning_check_devices, mode=mode)

In [ ]:
def jax_check_devices(mode="FLAT"):
    #logger.setLevel(logging.ERROR)
    os.environ["ZE_FLAT_DEVICE_HIERARCHY"] = mode

    # Set log level of root logger so as to avoid warnings.
    logging.basicConfig(level = logging.ERROR)   
    # Enable Intel Extension For OpenXLA profiler.
    # This suppresses warning that would otherwise be printed, even if profiler isn't used.
    os.environ["ZE_ENABLE_TRACING_LAYER"] = "1"
    os.environ["UseCyclesPerSecondTimer"] = "1"

    from jax._src.xla_bridge import backends
    print(f"{mode} mode - devices seen by jax:")
    for device_type in sorted(backends()):
        print(f"    {device_type}: {jax.devices(device_type)}") 
    print()

for mode in modes:
    run_in_subprocess(target=jax_check_devices, mode=mode)

In [ ]:
def torch_matrix_multiplication(mode="FLAT"):
    os.environ["ZE_FLAT_DEVICE_HIERARCHY"] = mode
    # Define device types to be considered.
    device_types = ["cpu", "cuda", "mps", "xpu", "fictional_device"]
    # Number of times to attempt matrix multiplication.
    n_attempt = 3
    # Print information about available device types.
    for device_type in device_types:
        # Determine number of devices of each type.
        try:
            device_module = importlib.import_module(f"torch.{device_type}")
        except ModuleNotFoundError:
            device_module = None
        if hasattr(device_module, "is_available") and device_module.is_available():
            n_device = device_module.device_count()
        else:
            n_device = 0
        print(f"\n{mode} mode - device type: {device_type}")
        print(f"Number of devices: {n_device}")
        # Test matrix-multiplication time for all devices of current type,
        # considering devices in random order.
        indices = list(range(n_device))
        random.shuffle(indices)
        i_dim = 0
        while n_device:
            dim = 2**i_dim
            i_dim += 1
            i_attempt = 0
            print()
            while i_attempt < n_attempt:
                i_attempt += 1
                for i_device in indices:
                    device_name = f"{device_type}:{i_device}"
                    if dim > 1024 and "cpu" == device_type:
                        n_device = 0
                        i_attempt = n_attempt + 1
                        break
                    t0 = time.time()
                    try:
                        x=torch.randn((dim, dim), device=torch.device(device_name))
                        y=torch.randn((dim, dim), device=torch.device(device_name))
                        z=torch.matmul(x,y)
                    except RuntimeError:
                        n_device =0
                    t1 = time.time()
                    if n_device:
                        print(f"{device_name}: order = {dim}; "
                                f"attempt ={i_attempt : 3d}; "
                                f"time ={(t1 - t0) * 1.e6 : 8.1f} microseconds")
                    else:
                        print(f"{device_type}: order = {dim}; out of memory")
                        i_attempt = n_attempt + 1
                        break

for mode in modes:
    run_in_subprocess(target=torch_matrix_multiplication, mode=mode)

xpu: order = 65536; out of memory

FLAT mode - device type: fictional_device
Number of devices: 0

COMPOSITE mode - device type: cpu
Number of devices: 1

cpu:0: order = 1; attempt =  1; time =   751.7 microseconds
cpu:0: order = 1; attempt =  2; time =    62.2 microseconds
cpu:0: order = 1; attempt =  3; time =    63.4 microseconds

cpu:0: order = 2; attempt =  1; time =    88.7 microseconds
cpu:0: order = 2; attempt =  2; time =    42.4 microseconds
cpu:0: order = 2; attempt =  3; time =    52.0 microseconds

cpu:0: order = 4; attempt =  1; time =    64.6 microseconds
cpu:0: order = 4; attempt =  2; time =    34.1 microseconds
cpu:0: order = 4; attempt =  3; time =    24.6 microseconds

cpu:0: order = 8; attempt =  1; time =    43.4 microseconds
cpu:0: order = 8; attempt =  2; time =    38.9 microseconds
cpu:0: order = 8; attempt =  3; time =    29.1 microseconds

cpu:0: order = 16; attempt =  1; time =    47.9 microseconds
cpu:0: order = 16; attempt =  2; time =    31.5 microseconds